# 01 Data Loading and Inspection

## Purpose
Document the project objective, inspect the raw data structure, and capture the initial quality issues that will shape later preprocessing and modeling decisions.

In [1]:
#======================
# Project Title: Outpatient No-Show Risk Model
# Objective: Identify scheduled outpatient appointments at elevated risk of no-show using information available before the visit so clinics can target reminder and re-engagement efforts.
# Data Source: Medical appointment no-show dataset stored in `Data/raw/noshows.csv`, where each row represents a scheduled appointment and the target is the `No-show` outcome.
# Methodology: Review raw appointment data, assess target balance and data quality, engineer pre-visit scheduling features, train interpretable baseline classification models, and evaluate performance for operational outreach use.
#=========================


In [2]:
from pathlib import Path

import pandas as pd


def print_section(title: str) -> None:
    print(f"\n{'=' * 12} {title} {'=' * 12}")


def summarize_frame(frame: pd.DataFrame) -> pd.DataFrame:
    summary = pd.DataFrame(
        {
            "dtype": frame.dtypes.astype(str),
            "missing_count": frame.isna().sum(),
            "missing_pct": frame.isna().mean().mul(100).round(2),
            "n_unique": frame.nunique(dropna=False),
        }
    )
    return summary.sort_values(["missing_pct", "n_unique"], ascending=[False, False])


## Project Setup
- Framed the task as predicting which scheduled outpatient appointments are at elevated risk of no-show.
- Defined the unit of analysis as a single appointment record rather than the patient across all visits.
- Treated the attendance outcome as the target and flagged any post-visit or operational follow-up fields for leakage review.

In [3]:
DATA_PATH = Path("Data/raw/noshows.csv")
TARGET_COLUMN = "No-show"

df = pd.read_csv(DATA_PATH)
df.head()


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [4]:
print_section("Shape and Preview")
print(df.shape)
display(df.sample(5, random_state=42))

print_section("Column Summary")
display(summarize_frame(df))


============ Shape and Preview ============
(110527, 14)

============ Column Summary ============


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
84674,2.759575e+13,5756417,F,2016-06-01T08:17:04Z,2016-06-01T00:00:00Z,20,ILHA DAS CAIEIRAS,0,0,0,0,0,0,No
3635,9.673194e+14,5523159,F,2016-03-29T16:09:39Z,2016-05-03T00:00:00Z,37,RESISTÊNCIA,0,0,0,0,0,1,No
16605,3.951641e+12,5693080,F,2016-05-12T17:33:56Z,2016-05-20T00:00:00Z,38,MARIA ORTIZ,0,0,0,0,0,0,Yes
103942,9.173245e+14,5654129,F,2016-05-03T13:54:51Z,2016-06-03T00:00:00Z,24,SANTO ANDRÉ,0,0,0,0,0,1,Yes
274,3.995366e+12,5641070,F,2016-04-29T12:16:28Z,2016-04-29T00:00:00Z,41,MARIA ORTIZ,0,0,0,0,0,0,No


,dtype,missing_count,missing_pct,n_unique
AppointmentID,int64,0,0.0,110527
ScheduledDay,object,0,0.0,103549
PatientId,float64,0,0.0,62299
Age,int64,0,0.0,104
Neighbourhood,object,0,0.0,81
AppointmentDay,object,0,0.0,27
Handcap,int64,0,0.0,5
Gender,object,0,0.0,2
Scholarship,int64,0,0.0,2
Hipertension,int64,0,0.0,2


In [5]:
print_section("Target Review")
if TARGET_COLUMN in df.columns:
    display(df[TARGET_COLUMN].value_counts(dropna=False))
    display(df[TARGET_COLUMN].value_counts(normalize=True, dropna=False).round(3))
else:
    print(f"Target column '{TARGET_COLUMN}' was not found.")


============ Target Review ============


No-show
No     88208
Yes    22319
Name: count, dtype: int64

No-show
No     0.798
Yes    0.202
Name: proportion, dtype: float64

In [6]:
print_section("Missingness Overview")
missingness = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
)
display(missingness[missingness > 0].to_frame())


============ Missingness Overview ============


,missing_pct


## Initial Data Quality Notes
- The raw file contains 110,527 appointments and 14 columns, with no missing values detected in the available fields.
- `PatientId` and `AppointmentID` behave like identifiers rather than clinically meaningful predictors and should be excluded from modeling.
- `ScheduledDay` and `AppointmentDay` are useful for engineered scheduling features, but their raw timestamp form would create leakage or high-cardinality noise if modeled directly.

## Takeaway Summary
The dataset is well structured for an appointment-level no-show prediction problem and the target is moderately imbalanced, with about 20% of visits marked as no-shows. The main preparation priority is not missing-data repair, but careful feature design that turns scheduling timestamps into usable pre-visit signals while excluding identifiers. That setup supports a realistic healthcare operations use case in which clinics act before the appointment occurs.